# 1 — Stage 1 machine setup and A100 preflight
Run one cell at a time. This notebook does not launch episodes. It clones the frozen branch, creates separate standard-LIBERO and LIBERO-Plus environments, and selects one idle physical A100. Never paste credentials into a saved cell.


In [ ]:
import os, subprocess
from pathlib import Path
REPO=Path.home()/"async-vla-latency-bench"
if not REPO.exists(): subprocess.run(["git","clone","https://github.com/MaheshGouru/async-vla-latency-bench.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","stage3-new-36-seeds"],check=True)
subprocess.run(["git","-C",str(REPO),"status","--short","--branch"],check=True)


In [ ]:
# Inspect hardware. Continue only with an A100-SXM4-40GB.
subprocess.run(["nvidia-smi"],check=True)
out=subprocess.run(["nvidia-smi","--query-gpu=index,name,memory.used,memory.total,utilization.gpu,compute_mode","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout
print(out)
free=[]
for line in out.strip().splitlines():
    idx,name,used,total,util,mode=[x.strip() for x in line.split(",")]
    if "A100-SXM4-40GB" in name and int(used)<500 and int(util)<5: free.append(int(idx))
if not free: raise SystemExit("STOP: no idle A100; do not share a GPU for latency testing")
GPU=free[0]; (Path.home()/"stage1_gpu.txt").write_text(str(GPU)+"\n")
print("Frozen physical GPU:",GPU,"(CUDA will expose it as cuda:0)")


## Install isolated environments
Standard LIBERO and LIBERO-Plus both provide the `libero` package and must not coexist. The commands below mirror the two repository Dockerfiles. Installation and the LIBERO-Plus asset download can take substantial time.


In [ ]:
import sys
ID=Path.home()/"venv-stage1-id"; OOD=Path.home()/"venv-stage1-ood"
if not ID.exists(): subprocess.run([sys.executable,"-m","venv",str(ID)],check=True)
if not OOD.exists(): subprocess.run([sys.executable,"-m","venv",str(OOD)],check=True)
LEROBOT="2aba372b4e217cc47db28e0f836859b20d1456c9"
subprocess.run([str(ID/"bin/pip"),"install","--upgrade","pip","wheel","setuptools"],check=True)
subprocess.run([str(ID/"bin/pip"),"install",f"lerobot[pi,libero] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT}"],check=True)
subprocess.run([str(ID/"bin/pip"),"install","-e",str(REPO)],check=True)
subprocess.run([str(OOD/"bin/pip"),"install","--upgrade","pip","wheel","setuptools"],check=True)
subprocess.run([str(OOD/"bin/pip"),"install",f"lerobot[pi] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT}"],check=True)
subprocess.run([str(OOD/"bin/pip"),"install","robosuite==1.4.1","bddl==1.0.1","easydict==1.13","mujoco==3.7.0","matplotlib==3.10.8","Wand==0.6.13","scikit-image==0.25.2","gym==0.26.2","future","huggingface_hub","pyarrow","pandas","pytest"],check=True)
subprocess.run([str(OOD/"bin/pip"),"install","-e",str(REPO)],check=True)
print("Environments installed")


In [ ]:
# Clone the exact LIBERO-Plus revision. Asset installation follows Dockerfile.modal.libero_plus; run it before notebook 2 if assets are absent.
PLUS=Path.home()/"LIBERO-plus"; PLUS_SHA="4976dc3"
if not PLUS.exists(): subprocess.run(["git","clone","https://github.com/sylvestf/LIBERO-plus.git",str(PLUS)],check=True)
subprocess.run(["git","-C",str(PLUS),"checkout",PLUS_SHA],check=True)
subprocess.run([str(OOD/"bin/pip"),"install","--no-deps","-e",str(PLUS)],check=True)
print("LIBERO-Plus SHA:",subprocess.run(["git","-C",str(PLUS),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip())
print("IMPORTANT: ensure ~/LIBERO-plus/libero/libero/assets exists; if not, install the 6.4 GB assets exactly as documented in Dockerfile.modal.libero_plus before continuing.")


In [ ]:
# Unit tests and immutable identity. Paste this output back for review.
subprocess.run([str(OOD/"bin/python"),"-m","pytest","-q",str(REPO/"async_vla_benchmark/tests/test_stage1.py")],cwd=REPO,check=True)
print("benchmark SHA",subprocess.run(["git","-C",str(REPO),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip())
print("GPU",(Path.home()/"stage1_gpu.txt").read_text().strip())
